In [3]:
import requests
import pandas as pd
import re
import urllib
from tqdm import tqdm
from github_helper import from_github

# This notebook converts voting_id's to dataframes of votes for each period

In [34]:
df_voting_sessions = pd.read_csv(from_github("/voting-data/voting_sessions_enriched.csv"))
print(df_voting_sessions.groupby("Period").size())

Period
65     294
66    1269
67    1778
68    1727
69    2019
70    1838
71    1381
dtype: int64


We choose to drop period 65 as it only has 294 voting_sessions. Keeping it would lead to potentially very volatile calculations later, which might skew the analysis excessively

In [ ]:
def get_voting_sessions_in_period(df_voting_sessions, period):
    vote_ids = df_voting_sessions[df_voting_sessions['Period'] == period]['afstemning_id'].unique() #Do i need to convert to list?
    print(f"Found {len(vote_ids)} votes in period {period}")
    return vote_ids

request_session = requests.Session()

def get_voting_session_with_votes(afstemning_id, session = request_session):
    base_url = "https://oda.ft.dk/api/"
    all_votes = []
    skip = 0

    next_link = None

    while True:
        url = f"{base_url}Afstemning({afstemning_id})/Stemme"
        # print(f"Base URL without params: {url}")
        if next_link:
            response = session.get(next_link)
        else:
            # response = session.get(url, params=params) #Use session to reuse connections and make everything run faster
            response = session.get(url)
            # print(f"This is the URL with parameters: {response.url}")
            # response = session.get(url)

        if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
            print(f"HTTP error for {afstemning_id}: ", response.status_code)
            print("Response text:", response.text)
            return None
        else:
            try:
                data = response.json()
            except ValueError:
                print("Error: Response is not valid JSON")
                print("Response text:", response.text)
                return None
        

        # print("Original_data", data)
        votes = data.get('value')
        next_link = data.get("odata.nextLink")
        # print("Next_link is ", next_link)
        # print("Value", votes)
        # votes = contained_data
        if not votes:
            # print("not votes????")
            break
        all_votes.extend(votes)
        if not next_link: #'https://oda.ft.dk/api/Afstemning(9700)/Stemme?$skip=100
            # print("No link found")
            break

        skip += 100

    return all_votes 

afstemning_id = 9700
data_from_voting_session = get_voting_session_with_votes(afstemning_id)


In [ ]:
def get_and_save_votes_from(df_voting_sessions, voting_period):
    afstemning_ids = get_voting_sessions_in_period(df_voting_sessions, voting_period)
    all_votes_full_period = []
    sessions_w_wrong_number_votes = []
    for afstemning_id in tqdm(afstemning_ids):
        votes_in_voting_session = get_voting_session_with_votes(afstemning_id)
        if len(votes_in_voting_session) != 179:
            sessions_w_wrong_number_votes.append((voting_period, afstemning_id, len(votes_in_voting_session)))
        all_votes_full_period.extend(votes_in_voting_session)
    
    df = pd.DataFrame(all_votes_full_period)

    #Rename the columns to something usefull
    df.rename(columns={"id": "vote_id"
                       ,"typeid": "vote_typeid"
                       ,"afstemningid": "afstemning_id"
                       ,"opdateringsdato" : "vote_opdateringsdato"
                       ,"aktørid" : "aktørid"
                       }
                       , inplace= True
                       )

    df.to_csv(f"./voting-data/df_votes_p{voting_period}.csv", index = False)
    unique_actors_in_period = df['aktørid'].unique()
    print(f"Found {len(unique_actors_in_period)} unique actors in period {voting_period}")
    return df, sessions_w_wrong_number_votes

all_sessions_with_wrong_number_votes = []
for voting_period in df_voting_sessions['Period'].unique():
    df, sessions_w_wrong_number_votes = get_and_save_votes_from(df_voting_sessions, voting_period)
    all_sessions_with_wrong_number_votes.extend(sessions_w_wrong_number_votes)
    
df_wrong = pd.DataFrame(all_sessions_with_wrong_number_votes, columns = ["Period", "afstemning_id", "n_votes"])
df.head()

Found 294 votes in period 65


100%|██████████| 294/294 [00:46<00:00,  6.35it/s]


Found 189 unique actors in period 65


,vote_id,vote_typeid,afstemning_id,aktørid,vote_opdateringsdato
0,1481475,3,5973,1886,2021-01-28T21:27:39.627
1,1481476,1,5973,1039,2018-02-16T10:35:23
2,1481477,1,5973,220,2018-02-16T10:35:23
3,1481478,1,5973,183,2018-02-16T10:35:23
4,1481479,1,5973,198,2018-02-16T10:35:23


We´ve manually checked the voting_sessions which had the wrong number of votes in a voting session, and they are all in accordance with what is available online. The errors fall into 3 categories: 1\) Between 177 and 182; We consider them representative 2) votes are 0. They will not be counted into our later calculations anyway 3) There is one observation with 144 votes. This is in accordance with what is available online. Since it's only one observation, we just leave it, and assume that it will not skew our analysis too much.

In [37]:
df = pd.DataFrame()
for period in [65, 66, 67, 68, 69, 70, 71]:
    df_votes_from_period = pd.read_csv(from_github(f"/voting-data/df_votes_p{period}.csv"))
    df = pd.concat([df, df_votes_from_period])

df.to_csv(f"./voting-data/df_votes_all_periods.csv", index = False)